# Analyzing the symmetry of contacts in an ENM

Motivation:

Up until now, an "ASU" description of an enm (e.g. `results/7tx0-enm-asu.csv`) actually contains symmetry-related copies of contacts. The thing that makes them asymmetric is just that contact involves an atom in the ASU. If each bond could be given a unique identifier, which would not change if the identities of the atoms were reversed, then it would be much easier to construct a minimal ASU description, and later to symmetry expand it.

Plan:

The first pass is to perform a symmetry analysis of a standard table.

1. Load `7tx0-enm-asu.csv` and the actual operators from `7tx0.cif` (following exactly the unit cell packing procedure that generated the csv).
2. For simplicity, index on the columns for group, op, and pbc_shift. Drop duplicates. (there will be one bond per interface).
3. For each row, compute its inverse image in the ASU (where the identities of the atoms are swapped).
4. Define an inequality on contact descriptions (op2, pbc_shift2).
5. Loop over rows: If inverse contact < contact, flip the atom identities. If inverse contact == contact (same ASU, or 2-fold axis), flip if doing so puts groups in ascending order (or if same group, flip if CRAs become asecnding).

In [1]:
import gemmi
from goodvibes import enm

coordinate_file = 'test_data/7TX0.cif'

st = gemmi.read_structure(coordinate_file)
st.setup_entities()  # supposed to be good practice
enm._pack_unit_cell(st, inplace=True)

<gemmi.Structure 7TX0 with 1 model(s)>

In [2]:
import pandas as pd
import ast

df = pd.read_csv("results/7tx0-enm-asu.csv",
                 converters={'pbc_shift1': lambda x: tuple(ast.literal_eval(x)),
                             'pbc_shift2': lambda x: tuple(ast.literal_eval(x))})
#is_redundant = df.set_index(['sym_idx2','pbc_shift2']).index.duplicated(keep='first')
#df = df[~is_redundant].reset_index(drop=True)
df

,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2,group_id1,group_id2
0,A/GLY 8/C,A/PHE 156/CZ,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
1,A/GLY 8/O,A/PHE 156/CE2,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
2,A/GLY 8/O,A/PHE 156/CZ,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
3,A/TYR 9/C,A/PHE 156/CZ,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
4,A/TYR 9/O,A/PHE 156/CE2,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
...,...,...,...,...,...,...,...,...
612,B/LEU 169/CD1,B/ASP 22/OD1,0,1,"(0, 0, 0)","(-1, 0, -1)",1,1
613,B/LEU 169/CD1,B/ASP 22/OD2,0,1,"(0, 0, 0)","(-1, 0, -1)",1,1
614,B/LEU 169/CD2,B/ILE 23/CG2,0,1,"(0, 0, 0)","(-1, 0, -1)",1,1
615,B/LEU 169/CD2,B/ALA 52/CB,0,1,"(0, 0, 0)","(-1, 0, -1)",1,1


In [3]:
import re

def invert_symop(st, sym_idx, pbc_shift):
    if sym_idx == 0 and pbc_shift == (0, 0, 0):
        return (0, (0, 0, 0))
    t = enm._symop_to_transform(st.cell.images, sym_idx, pbc_shift)
    t_inverse = t.inverse()
    return enm._transform_to_symop(st.cell.images, t_inverse)

def symop_to_key(sym_idx, pbc_shift):
    """make an expanded key for sorting purposes"""
    pbc_abs_sum = abs(pbc_shift[0]) + abs(pbc_shift[1]) + abs(pbc_shift[2])
    return (pbc_abs_sum, -1*pbc_shift[0], -1*pbc_shift[1], -1*pbc_shift[2], sym_idx)

pattern = re.compile(r'([^/]+)/([^ ]+) (\d+)/([^\.]+)(?:\.(.*))?')

def cra2key(cra):
    """make a key for sorting based on cra"""
    # for example; A/LYS 76/CD.A
    # use regex to split as {chain}/{resname} {resnum}/{atom}.{alt} where .{alt} is optional
    match = pattern.match(cra)
    if not match:
        raise ValueError(f"Invalid cra format: {cra}")
    chain, resname, resnum, atom, alt = match.groups()
    return (chain, int(resnum), resname, atom, alt)

# use itertuples to loop over rows
for row in df.itertuples():
    sym_idx, pbc_shift = invert_symop(st, row.sym_idx2, row.pbc_shift2)
    if sym_idx == row.sym_idx2 and pbc_shift == row.pbc_shift2:
        if row.group_id1 == row.group_id2:
            swap = cra2key(row.cra2) < cra2key(row.cra1)
        else:
            swap = row.group_id2 < row.group_id1
    elif symop_to_key(sym_idx, pbc_shift) < symop_to_key(row.sym_idx2, row.pbc_shift2):
        swap = True
    else:
        swap = False
    if swap:
        df.at[row.Index, 'sym_idx2'] = sym_idx
        df.at[row.Index, 'pbc_shift2'] = pbc_shift
        df.at[row.Index, 'cra1'] = row.cra2
        df.at[row.Index, 'cra2'] = row.cra1
        df.at[row.Index, 'group_id1'] = row.group_id2
        df.at[row.Index, 'group_id2'] = row.group_id1
df


,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2,group_id1,group_id2
0,A/GLY 8/C,A/PHE 156/CZ,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
1,A/GLY 8/O,A/PHE 156/CE2,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
2,A/GLY 8/O,A/PHE 156/CZ,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
3,A/TYR 9/C,A/PHE 156/CZ,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
4,A/TYR 9/O,A/PHE 156/CE2,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
...,...,...,...,...,...,...,...,...
612,B/ASP 22/OD1,B/LEU 169/CD1,0,3,"(0, 0, 0)","(0, -1, 0)",1,1
613,B/ASP 22/OD2,B/LEU 169/CD1,0,3,"(0, 0, 0)","(0, -1, 0)",1,1
614,B/ILE 23/CG2,B/LEU 169/CD2,0,3,"(0, 0, 0)","(0, -1, 0)",1,1
615,B/ALA 52/CB,B/LEU 169/CD2,0,3,"(0, 0, 0)","(0, -1, 0)",1,1


In [4]:
# number of duplicated rows (where all the columns are identical)
df.duplicated().sum()

np.int64(267)

In [5]:
# drop the duplicated rows (where all the columns are identical)
df = df.drop_duplicates().reset_index(drop=True)
df

,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2,group_id1,group_id2
0,A/GLY 8/C,A/PHE 156/CZ,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
1,A/GLY 8/O,A/PHE 156/CE2,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
2,A/GLY 8/O,A/PHE 156/CZ,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
3,A/TYR 9/C,A/PHE 156/CZ,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
4,A/TYR 9/O,A/PHE 156/CE2,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
...,...,...,...,...,...,...,...,...
345,B/ASP 157/OD2,B/ASP 162/O,0,3,"(0, 0, 0)","(0, -1, 0)",1,1
346,B/ASP 157/OD2,B/SER 166/N,0,3,"(0, 0, 0)","(0, -1, 0)",1,1
347,B/ASP 157/OD2,B/SER 166/CA,0,3,"(0, 0, 0)","(0, -1, 0)",1,1
348,B/ASP 157/OD2,B/SER 166/CB,0,3,"(0, 0, 0)","(0, -1, 0)",1,1


In [6]:
# now, sort the rows using our keys for symop2 and cra1
df["_sort_key"] = df.apply(
    lambda r: (symop_to_key(r.sym_idx2, r.pbc_shift2), cra2key(r.cra1)),
    axis=1,
)

df = df.sort_values("_sort_key").drop(columns="_sort_key").reset_index(drop=True)
df

,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2,group_id1,group_id2
0,A/LYS 76/CD.A,B/GLN 118/O,0,0,"(0, 0, 0)","(0, 0, 0)",0,1
1,A/LYS 76/CD.B,B/GLN 118/O,0,0,"(0, 0, 0)","(0, 0, 0)",0,1
2,A/LYS 76/CE.A,B/GLN 118/O,0,0,"(0, 0, 0)","(0, 0, 0)",0,1
3,A/LYS 76/CE.B,B/GLN 118/O,0,0,"(0, 0, 0)","(0, 0, 0)",0,1
4,A/LYS 76/NZ.A,B/HIS 119/ND1,0,0,"(0, 0, 0)","(0, 0, 0)",0,1
...,...,...,...,...,...,...,...,...
345,B/ASP 157/OD2,B/ASP 162/O,0,3,"(0, 0, 0)","(0, -1, 0)",1,1
346,B/ASP 157/OD2,B/SER 166/N,0,3,"(0, 0, 0)","(0, -1, 0)",1,1
347,B/ASP 157/OD2,B/SER 166/CA,0,3,"(0, 0, 0)","(0, -1, 0)",1,1
348,B/ASP 157/OD2,B/SER 166/CB,0,3,"(0, 0, 0)","(0, -1, 0)",1,1
